In [2]:
import os
import io
import zipfile
import pandas as pd
from PIL import Image
from tqdm import tqdm
import glob

def process_images_only(df, zip_name):
    save_path = f'/kaggle/working/{zip_name}'
    csv_save_path = save_path.replace('.zip', '.csv')
    processed_records = []
    
    print(f"\n--- {zip_name} 이미지 리사이즈 시작 ---")
    with zipfile.ZipFile(save_path, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as img_zip:
        for i, row in tqdm(df.iterrows(), total=len(df)):
            img_path = row['Full_Path']
            
            # 파일명을 유니크하게 생성 (나중에 학습 시 식별자 역할)
            unique_name = "_".join(img_path.split('/')[-4:]).replace('.png', '.jpg')
            
            try:
                with Image.open(img_path) as img:
                    # 1. 이미지 전처리 (320x320, RGB)
                    img = img.convert('RGB').resize((320, 320), Image.BILINEAR)
                    
                    # 2. 메모리 버퍼에 JPEG로 저장
                    buffer = io.BytesIO()
                    img.save(buffer, format="JPEG", quality=90)
                    
                    # 3. ZIP 파일에 추가
                    img_zip.writestr(unique_name, buffer.getvalue())
                    
                    # 4. 메타데이터 기록 (라벨은 원본 그대로 복사)
                    row_dict = row.to_dict()
                    row_dict['Zip_Entry_Name'] = unique_name
                    processed_records.append(row_dict)
            except Exception:
                continue

    # 결과 CSV 저장 (원본 라벨 값이 보존됨)
    new_df = pd.DataFrame(processed_records)
    new_df.to_csv(csv_save_path, index=False)
    print(f"완료: {save_path} / {csv_save_path}")

# --- 1. NIH 데이터 로드 (원본 유지) ---
NIH_BASE = '/kaggle/input/datasets/organizations/nih-chest-xrays/data/'
nih_df = pd.read_csv(os.path.join(NIH_BASE, 'Data_Entry_2017.csv'))
all_paths = {os.path.basename(x): x for x in glob.glob(os.path.join(NIH_BASE, 'images*', 'images', '*.png'))}
nih_df['Full_Path'] = nih_df['Image Index'].map(all_paths)

# NIH 분할 리스트 로드
with open(os.path.join(NIH_BASE, 'train_val_list.txt'), 'r') as f:
    train_list = [line.strip() for line in f.readlines()]

nih_train = nih_df[nih_df['Image Index'].isin(train_list)]
nih_val = nih_df[~nih_df['Image Index'].isin(train_list)]

# process_images_only(nih_train, 'nih_train_320.zip')
process_images_only(nih_val, 'nih_val_320.zip')

# --- 2. CheXpert 데이터 로드 (원본 라벨 -1, NaN 그대로 유지) ---
CH_BASE = '/kaggle/input/datasets/ashery/chexpert/'
ch_train = pd.read_csv(os.path.join(CH_BASE, 'train.csv'))
ch_val = pd.read_csv(os.path.join(CH_BASE, 'valid.csv'))

for df_tmp in [ch_train, ch_val]:
    # 이미지 경로만 매핑 (라벨 가공 코드 제거)
    df_tmp['Path'] = df_tmp['Path'].str.replace("CheXpert-v1.0-small/","")
    df_tmp['Full_Path'] = df_tmp['Path'].apply(lambda x: os.path.join(CH_BASE, x.strip('/')))

# process_images_only(ch_train, 'chexpert_train_320.zip')
process_images_only(ch_val, 'chexpert_val_320.zip')

print(len(nih_train),len(nih_val),len(ch_train),len(ch_val))

NameError: name 'zip_and_save_csv' is not defined